# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
 level=logging.INFO,
 format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
 datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
 SparkSession
 .builder
 .appName("SessaoLocal")
 .master("local[12]")
 .config("spark.driver.memory", "24g")
 .getOrCreate()
)
spark

## Arquivos de dados

### Dados de origem

In [ ]:
folder_claro = Path("/assets/monitoring/cdr/cdr_bruto/Semana101/Claro/home")
_files_claro = [(str(file), str(file.parent.name)) for file in folder_claro.rglob("*AXE/*.gz") if file.is_file()]
df = pd.DataFrame(_files_claro, columns=["file_path", "parent_folder"])
df["exchange_id"] = df["parent_folder"].str.slice(stop=4)
files_claro = df.groupby("exchange_id").sample(1,random_state=42)["file_path"].to_list()
files_claro[:5]

### Arquivos intermediário e final

In [ ]:
extracted_claro = "/assets/data/cdr/teletools/db/extraidos/fcdr_claro_extraido.parquet"
transformed_claro = "/assets/data/cdr/teletools/db/transformados/fcdr_claro_transformado.parquet"

# Análise

## Dados brutos

In [ ]:
df_raw = spark.read.csv(files_claro, sep=";", lineSep="<", schema="value string")
dfx = df_raw.limit(5).toPandas()
dfx

In [ ]:
df_raw.show(5,truncate=False)

In [ ]:
column_indices = (1, 9, 12, 14, 32, 33, 35, 59, 61, 67, 73, 79, 85, 86, 88, 91, 95, 99, 102, 106, 110, 114, 117, 118, 122, 126, 128, 131, 133, 134, 137, 139, 140, 141, 149)
column_sizes = (8, 3, 2, 18, 1, 2, 24, 2, 6, 6, 6, 6, 1, 2, 3, 4, 4, 3, 4, 4, 4, 3, 1, 5, 3, 2, 3, 2, 1, 3, 2, 1, 1, 8, 8)
column_names = ("Nome da Central", "Tamanho do Registro", "Tipo de Registro", "Assinante A", "Parte Tarifada", "Categoria de A", "Assinante B", "Categoria de B (Fim de Seleção-FDS)", "Hora de Início de Tarifação - 24h", "Hora de Término de Tarifação", "Duração Tarifável", "Data de Início de Tarifação", "Causa da Saída", "Contador de Saídas Parciais", "Reserva", "Número da Rota de Entrada", "Número do Juntor de Entrada", "Reserva", "Número da Rota de Saída", "Número do Juntor de Saída", "Caso de Tarifação", "Classe de Tarifa", "Estado da Chamada", "Código da Falha (EOS)", "Código do Serv. De Telecom.", "Tipo de Assinante A", "Tipo de Número de A", "Plano de Numeração de A", "Tipo de Assinante B", "Tipo de Número de B", "Indicador do Plano de Numeração de B", "Indicador de Chamada de Conferencia", "Indicador de re-roteamento", "Número de Identificação da Chamada", "Número de Seqüência do CDR")

columns_expressions = [
    F.trim(
        F.regexp_replace(F.substring(F.col("value"), indice, size), "-", "")
    ).alias(name)
    for indice, size, name in zip(
        column_indices, column_sizes, column_names
    )
]
df = df_raw.select(*columns_expressions)
df.show(5)

In [ ]:
columns_to_keep = [
    "Nome da Central", 
    "Tipo de Registro", 
    "Assinante A", 
    "Assinante B", 
    "Categoria de B (Fim de Seleção-FDS)", 
    "Hora de Início de Tarifação - 24h", 
    "Hora de Término de Tarifação", 
    "Duração Tarifável", 
    "Data de Início de Tarifação", 
    "Número da Rota de Entrada", 
    "Número da Rota de Saída", 
    "Número de Identificação da Chamada",
]
df.select(columns_to_keep).show(5)

In [ ]:
axe_layout = [(column_indices, column_sizes, column_names) for column_indices, column_sizes, column_names in zip(column_indices, column_sizes, column_names)]
axe_layout_selected = [field for field in axe_layout if field[2] in columns_to_keep]
axe_layout_selected

In [ ]:
# df.groupBy(F.col("Nome da Central")).count().show()

In [ ]:
# df.groupBy(F.col("Tipo de Registro")).count().show()

In [ ]:
# df.groupBy(F.col("Categoria de B (Fim de Seleção-FDS)")).count().show()

In [ ]:
# df.groupBy(F.col("Duração Tarifável")).count().show()

## Extração

In [ ]:
extractor = CDRTextExtractor(spark)
extractor.extract(files_claro, extracted_claro, "stfc_axe_claro")

In [ ]:
df_extracted = spark.read.parquet(extracted_claro)
df_extracted.show(5, truncate=False)

## Transformação

In [ ]:
transformer = CDRTransformer(spark)
transformer.transform_stfc_axe_claro(extracted_claro, transformed_claro)

In [ ]:
df_transformed = spark.read.parquet(transformed_claro)
df_transformed.limit(5).toPandas().dropna(how="all",axis=1)